In [ ]:
import pandas as pd
import numpy as np

# 1. Memuat Dataset
# Pastikan file ADNI_POMDP_Complete.csv berada di folder yang sama dengan skrip ini
print("Memuat dataset...")
df = pd.read_csv('ADNI_POMDP_Complete.csv')

# 2. DATA CLEANING (Pembersihan Data)
print(f"Jumlah data awal: {len(df)} baris")

# Menghapus baris yang tidak memiliki label Diagnosis (DX) atau hasil tes MMSE
df_clean = df.dropna(subset=['DX', 'MMSE']).copy()
print(f"Jumlah data setelah dibersihkan dari NA: {len(df_clean)} baris")

# 3. DISKRITISASI (Sesuai Metode Riset Tahap 1)
# Mengubah skor MMSE numerik menjadi format kategori untuk model POMDP/DESPOT
# Aturan umum: > 24 (Normal), 20-24 (Sedang/MCI), < 20 (Rendah/Demensia)
def discretize_mmse(score):
    if score > 24:
        return 'Normal'
    elif score >= 20:
        return 'Sedang'
    else:
        return 'Rendah'

df_clean['MMSE_Category'] = df_clean['MMSE'].apply(discretize_mmse)

print("\nDistribusi Observasi (MMSE Kategori):")
print(df_clean['MMSE_Category'].value_counts())

# 4. MENGHITUNG PROBABILITAS OBSERVASI (Z)
# P(Observation | State) -> Probabilitas hasil tes MMSE jika pasien berada di kondisi DX tertentu
print("\n--- MATRIKS PROBABILITAS OBSERVASI (Z) ---")
# Menghitung persentase kategori MMSE berdasarkan setiap Diagnosis
obs_prob = pd.crosstab(df_clean['DX'], df_clean['MMSE_Category'], normalize='index')
print(obs_prob.round(3))
# (Hasil ini nanti dimasukkan ke dalam algoritma DESPOT)


# 5. MENGHITUNG PROBABILITAS TRANSISI (T)
# P(State' | State) -> Perubahan penyakit seiring waktu untuk pasien (RID) yang sama
print("\n--- MATRIKS PROBABILITAS TRANSISI (T) ---")

# Mengurutkan data berdasarkan Pasien (RID) dan Tanggal Kunjungan (DATE)
df_clean = df_clean.sort_values(by=['RID', 'DATE'])

# Membuat kolom baru yang berisi Diagnosis pada kunjungan "selanjutnya" (Shift -1)
df_clean['Next_DX'] = df_clean.groupby('RID')['DX'].shift(-1)

# Membuang baris yang 'Next_DX'-nya kosong (artinya itu adalah kunjungan terakhir pasien)
df_transition = df_clean.dropna(subset=['Next_DX'])

# Menghitung probabilitas transisi dari DX saat ini ke Next_DX
trans_prob = pd.crosstab(df_transition['DX'], df_transition['Next_DX'], normalize='index')
print(trans_prob.round(3))
# (Hasil ini menunjukkan misal: Berapa % pasien MCI yang berubah menjadi Dementia)

print("\nPra-pemrosesan selesai! Matriks T dan Z siap digunakan untuk memodelkan DESPOT.")

Memuat dataset...
Jumlah data awal: 15836 baris
Jumlah data setelah dibersihkan dari NA: 12464 baris

Distribusi Observasi (MMSE Kategori):
MMSE_Category
Normal    10090
Sedang     1707
Rendah      667
Name: count, dtype: int64

--- MATRIKS PROBABILITAS OBSERVASI (Z) ---
MMSE_Category  Normal  Rendah  Sedang
DX                                   
CN              0.991   0.000   0.009
Dementia        0.270   0.257   0.473
MCI             0.897   0.007   0.096

--- MATRIKS PROBABILITAS TRANSISI (T) ---
Next_DX      CN  Dementia    MCI
DX                              
CN        0.929     0.003  0.068
Dementia  0.001     0.976  0.023
MCI       0.035     0.108  0.857

Pra-pemrosesan selesai! Matriks T dan Z siap digunakan untuk memodelkan DESPOT.


In [ ]:
# Menyimpan data yang telah dibersihkan ke file CSV baru
file_name = 'ADNI_POMDP_Cleaned.csv'
df_clean.to_csv(file_name, index=False)

print(f"Data berhasil diekspor ke: {file_name}")
print(f"Jumlah baris dalam file: {len(df_clean)}")

Data berhasil diekspor ke: ADNI_POMDP_Cleaned.csv
Jumlah baris dalam file: 12464


In [ ]:
print(df_clean['DX'].value_counts(normalize=True))

DX
MCI         0.417041
CN          0.385189
Dementia    0.197770
Name: proportion, dtype: float64
